In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')
!pip install -q datasets transformers pandas scikit-learn

In [ ]:
import json
import re
import os
import pandas as pd
from datasets import load_dataset

print("All libraries imported successfully!")
print(f"Pandas version    : {pd.__version__}")

In [ ]:
# This downloads IMDb directly from Hugging Face (~80MB)
# Internet must be ON in Kaggle settings for this to work

print("Downloading IMDb dataset from Hugging Face...")
dataset = load_dataset("imdb")

# Convert splits to DataFrames
train_df = pd.DataFrame(dataset["train"])
test_df  = pd.DataFrame(dataset["test"])

print("\n✅ Dataset loaded successfully!")
print(f"Train size : {len(train_df):,} rows")
print(f"Test size  : {len(test_df):,} rows")
print(f"Columns    : {list(train_df.columns)}")

In [ ]:
# ── Size and structure ────────────────────────────────────────
print("=" * 50)
print("RAW DATA INSPECTION")
print("=" * 50)

print(f"\nShape of train set : {train_df.shape}")
print(f"Shape of test set  : {test_df.shape}")

# ── Class distribution ────────────────────────────────────────
print("\nLabel distribution in TRAIN set:")
print(train_df["label"].value_counts())
print(f"\nClass balance: {train_df['label'].value_counts(normalize=True).mul(100).round(1).to_dict()}")

# ── Missing values ────────────────────────────────────────────
print("\nMissing values in train:")
print(train_df.isnull().sum())

print("\nMissing values in test:")
print(test_df.isnull().sum())

# ── Sample raw review ─────────────────────────────────────────
print("\nSample RAW review (first 400 chars):")
print("-" * 40)
print(train_df["text"].iloc[0][:400])
print("-" * 40)
print(f"\nLabel : {train_df['label'].iloc[0]} ({'positive' if train_df['label'].iloc[0]==1 else 'negative'})")

In [ ]:
def clean_text(text):
    """
    Cleaning steps applied (justify each in your report):

    1. Lowercase      : Sentiment doesn't depend on case
    2. Remove HTML    : IMDb reviews contain raw <br /> tags
    3. Remove punctuation : Reduces noise for tokenizer
    4. Collapse whitespace: Normalises spacing after removals
    """
    text = text.lower()                          # 1. lowercase
    text = re.sub(r"<br\s*/?>", " ", text)       # 2. remove HTML <br> tags
    text = re.sub(r"[^a-z0-9\s]", "", text)      # 3. remove punctuation/symbols
    text = re.sub(r"\s+", " ", text).strip()     # 4. collapse extra whitespace
    return text


# Apply to both splits
print("Cleaning train set...")
train_df["text"] = train_df["text"].apply(clean_text)

print("Cleaning test set...")
test_df["text"] = test_df["text"].apply(clean_text)

# Show before vs after comparison
print("\n✅ Cleaning complete!")
print("\nSample CLEANED review (first 400 chars):")
print("-" * 40)
print(train_df["text"].iloc[0][:400])
print("-" * 40)

In [ ]:
# ── Train set ─────────────────────────────────────────────────
before_train = len(train_df)
train_df.drop_duplicates(subset="text", inplace=True)
train_df.dropna(subset=["text", "label"], inplace=True)
after_train = len(train_df)

# ── Test set ──────────────────────────────────────────────────
before_test = len(test_df)
test_df.drop_duplicates(subset="text", inplace=True)
test_df.dropna(subset=["text", "label"], inplace=True)
after_test = len(test_df)

print("=" * 40)
print("CLEANING SUMMARY")
print("=" * 40)
print(f"Train : {before_train:,} → {after_train:,} rows ({before_train - after_train} removed)")
print(f"Test  : {before_test:,}  → {after_test:,}  rows ({before_test - after_test} removed)")

print("\nLabel distribution AFTER cleaning (train):")
print(train_df["label"].value_counts())

In [ ]:
import json

# Define label mappings
id2label = {"0": "negative", "1": "positive"}
label2id = {"negative": 0,   "positive": 1}

mapping = {
    "id2label": id2label,
    "label2id": label2id
}

# Save to Kaggle working directory
os.makedirs("/kaggle/working/data", exist_ok=True)

with open("/kaggle/working/data/id2label.json", "w") as f:
    json.dump(mapping, f, indent=2)

print("✅ Saved: /kaggle/working/data/id2label.json")
print("\nContents:")
print(json.dumps(mapping, indent=2))

In [ ]:
# Save cleaned data to Kaggle working directory
train_df.to_csv("/kaggle/working/data/train_clean.csv", index=False)
test_df.to_csv("/kaggle/working/data/test_clean.csv",   index=False)

print("✅ Saved: /kaggle/working/data/train_clean.csv")
print("✅ Saved: /kaggle/working/data/test_clean.csv")

# Verify file sizes
import os
train_size = os.path.getsize("/kaggle/working/data/train_clean.csv") / (1024*1024)
test_size  = os.path.getsize("/kaggle/working/data/test_clean.csv")  / (1024*1024)

print(f"\nFile sizes:")
print(f"  train_clean.csv : {train_size:.1f} MB")
print(f"  test_clean.csv  : {test_size:.1f} MB")
print("\n⚠️  Note: These CSVs stay on Kaggle only.")
print("Only id2label.json gets committed to GitHub.")

In [ ]:
!pip install -q transformers datasets wandb scikit-learn accelerate huggingface_hub
print('✅ Dependencies installed.')

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
HF_TOKEN = secrets.get_secret('HF_TOKEN')

import wandb
from huggingface_hub import login

login(token=HF_TOKEN)
wandb.login()
print('✅ Logged in to W&B and Hugging Face.')

In [ ]:
# ── Shared setup: load cleaned data + id2label from Task 2 ──────────────────
import json
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, f1_score

# Load CSVs saved by Task 2
train_df = pd.read_csv('/kaggle/working/data/train_clean.csv')
test_df  = pd.read_csv('/kaggle/working/data/test_clean.csv')

# Subsample to stay within Kaggle free GPU limits
TRAIN_SIZE = 10_000
EVAL_SIZE  =  2_000

train_df = train_df.sample(n=TRAIN_SIZE, random_state=42).reset_index(drop=True)
test_df  = test_df.sample(n=EVAL_SIZE,  random_state=42).reset_index(drop=True)

print(f'Train size : {len(train_df):,}')
print(f'Eval size  : {len(test_df):,}')
print(f'Label distribution (train):\n{train_df["label"].value_counts()}')

# Load id2label from Task 2 (string keys -> convert to int)
with open('/kaggle/working/data/id2label.json') as f:
    mapping = json.load(f)

id2label = {int(k): v for k, v in mapping['id2label'].items()}
label2id = {v: int(k) for k, v in mapping['id2label'].items()}

print('\nid2label:', id2label)
print('label2id:', label2id)

In [ ]:
# ── Tokenize once — both versions share the same tokenized datasets ──────────
MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
print(f'✅ Tokenizer loaded: {MODEL_NAME}')

train_hf = Dataset.from_pandas(train_df[['text', 'label']])
eval_hf  = Dataset.from_pandas(test_df[['text', 'label']])

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=256)

train_hf = train_hf.map(tokenize, batched=True)
eval_hf  = eval_hf.map(tokenize,  batched=True)

train_hf = train_hf.rename_column('label', 'labels')
eval_hf  = eval_hf.rename_column('label',  'labels')

train_hf.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
eval_hf.set_format('torch',  columns=['input_ids', 'attention_mask', 'labels'])

print('✅ Tokenization complete.')
print(f'   Train: {train_hf}')
print(f'   Eval : {eval_hf}')

In [ ]:
# ── Helper: run one experiment version ───────────────────────────────────────
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1':       f1_score(labels, preds, average='weighted'),
    }

def run_experiment(version, learning_rate, epochs=3, batch_size=16,
                   warmup_steps=200, weight_decay=0.01):
    """
    Fine-tunes DistilBERT for one experiment version.
    Returns (trainer, results_dict) so the best model can be pushed in Task 5.
    """
    print(f'\n{"="*55}')
    print(f'  Starting {version}  |  LR={learning_rate}  |  Epochs={epochs}')
    print(f'{"="*55}\n')

    # Fresh model for each run (no weight leakage between versions)
    model = DistilBertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=id2label,
        label2id=label2id,
    )

    wandb.init(
        project='mlops-assignment3',
        name=f'run-{version}',
        config={
            'model_name':    MODEL_NAME,
            'epochs':        epochs,
            'batch_size':    batch_size,
            'learning_rate': learning_rate,
            'warmup_steps':  warmup_steps,
            'weight_decay':  weight_decay,
            'version':       version,
            'platform':      'Kaggle',
            'dataset':       'IMDb',
        },
    )

    args = TrainingArguments(
        output_dir                  = f'/kaggle/working/results-{version}',
        num_train_epochs            = epochs,
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = batch_size,
        learning_rate               = learning_rate,
        warmup_steps                = warmup_steps,
        weight_decay                = weight_decay,
        eval_strategy               = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'f1',
        report_to                   = 'wandb',
        run_name                    = f'run-{version}',
        logging_steps               = 50,
        fp16                        = True,
    )

    trainer = Trainer(
        model           = model,
        args            = args,
        train_dataset   = train_hf,
        eval_dataset    = eval_hf,
        compute_metrics = compute_metrics,
    )

    trainer.train()
    results = trainer.evaluate()

    print(f'\n✅ {version} results: {results}')

    wandb.run.summary['final_accuracy'] = results['eval_accuracy']
    wandb.run.summary['final_f1']       = results['eval_f1']
    wandb.run.summary['final_loss']     = results['eval_loss']
    wandb.finish()

    return trainer, results

print('✅ run_experiment() defined.')

In [ ]:
trainer_v1, results_v1 = run_experiment(
    version       = 'v1',
    learning_rate = 3e-5,
)

In [ ]:
trainer_v2, results_v2 = run_experiment(
    version       = 'v2',
    learning_rate = 5e-5,
)

In [ ]:
# ── Compare v1 vs v2 ─────────────────────────────────────────────────────────
print('\n' + '='*55)
print('  EXPERIMENT COMPARISON')
print('='*55)
print(f'{"Metric":<20} {"v1 (LR=3e-5)":>15} {"v2 (LR=5e-5)":>15}')
print('-'*55)
print(f'{"Accuracy":<20} {results_v1["eval_accuracy"]:>15.4f} {results_v2["eval_accuracy"]:>15.4f}')
print(f'{"F1 (weighted)":<20} {results_v1["eval_f1"]:>15.4f} {results_v2["eval_f1"]:>15.4f}')
print(f'{"Val Loss":<20} {results_v1["eval_loss"]:>15.4f} {results_v2["eval_loss"]:>15.4f}')
print('='*55)

best_version = 'v1' if results_v1['eval_f1'] >= results_v2['eval_f1'] else 'v2'
best_trainer = trainer_v1 if best_version == 'v1' else trainer_v2
print(f'\n🏆 Best version: {best_version}')

In [ ]:
# ── Task 5: Push best model to Hugging Face Hub ───────────────────────────────

HF_REPO_ID   = 'pjayG25AIT2045/distilbert-imdb-mlops'
HF_MODEL_URL = f'https://huggingface.co/{HF_REPO_ID}'

print(f'Pushing best model ({best_version}) to: {HF_MODEL_URL}')

best_trainer.model.push_to_hub(HF_REPO_ID)
tokenizer.push_to_hub(HF_REPO_ID)
print('✅ Push complete.')

# Log HF model URL into W&B run summary
wandb.init(
    project = 'mlops-assignment3',
    name    = f'model-push-{best_version}',
    resume  = 'allow',
)
wandb.run.summary['huggingface_model'] = HF_MODEL_URL
wandb.finish()

print(f'✅ HF model URL logged to W&B: {HF_MODEL_URL}')